# Forward and Backward Pass by Hand, Then in NumPy

Notebook for **Chapter 2** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch02.html

Tip: change a number, predict the output, then run the cell again.

## 2.1 The Forward Pass and the Computation Graph

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
sig = lambda z: 1/(1+np.exp(-z))
x = np.array([1., 2.]); y = 1.
W1 = np.array([[0.1,0.2],[0.3,0.4]]); b1 = np.array([0.1,-0.1])
W2 = np.array([0.5,-0.6]); b2 = 0.2
def fwd(W1,b1,W2,b2):
    z1 = W1@x + b1; h = sig(z1); z2 = W2@h + b2; p = sig(z2)
    L = -(y*np.log(p) + (1-y)*np.log(1-p))
    return z1,h,z2,p,L
z1,h,z2,p,L = fwd(W1,b1,W2,b2)
print("z1",z1,"h",h,"z2",z2,"p",p,"L",L)

z1 [0.6 1. ] h [0.6457 0.7311] z2 0.08419300593489476 p 0.521035826979376 L 0.6519364737976816


## 2.2 The Backward Pass: Sending the Blame Backward

In [ ]:
dz2 = p - y
dW2 = dz2*h; db2 = dz2
dh  = dz2*W2
dz1 = dh*h*(1-h)
dW1 = np.outer(dz1, x); db1 = dz1
print("dz2",dz2,"dW2",dW2,"dh",dh,"dz1",dz1)
print("dW1",dW1,"db1",db1)

dz2 -0.478964173020624 dW2 [-0.3092 -0.3502] dh [-0.2395  0.2874] dz1 [-0.0548  0.0565]
dW1 [[-0.0548 -0.1096]
 [ 0.0565  0.113 ]] db1 [-0.0548  0.0565]


## 2.3 Gradient Checking and One Update Step

In [ ]:
eps = 1e-6
W1p = W1.copy(); W1p[0,1] += eps
print("num dW1[0,1]", (fwd(W1p,b1,W2,b2)[4]-L)/eps, "analytic", dW1[0,1])

num dW1[0,1] -0.1095794159855501 analytic -0.10957945453047446


In [ ]:
lr = 0.5
W1n = W1 - lr*dW1; b1n = b1 - lr*db1; W2n = W2 - lr*dW2; b2n = b2 - lr*db2
print("W1n",W1n,"b1n",b1n,"W2n",W2n,"b2n",b2n)
print("new L", fwd(W1n,b1n,W2n,b2n)[4])

W1n [[0.1274 0.2548]
 [0.2717 0.3435]] b1n [ 0.1274 -0.1283] W2n [ 0.6546 -0.4249] b2n 0.439482086510312
new L 0.4409751567165639


## 2.4 A Complete Neural Network in NumPy

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
# two moons dataset made by hand
n = 100
t = rng.uniform(0, np.pi, n)
a = np.c_[np.cos(t), np.sin(t)]; b = np.c_[1-np.cos(t), 0.5-np.sin(t)]
X = np.vstack([a, b]) + rng.normal(0, 0.1, (2*n, 2))
y = np.r_[np.zeros(n), np.ones(n)].astype(int)

class MLP:
    def __init__(s, sizes, seed=0):
        r = np.random.default_rng(seed)
        s.W = [r.normal(0, np.sqrt(2/i), (i, o)) for i, o in zip(sizes[:-1], sizes[1:])]
        s.b = [np.zeros(o) for o in sizes[1:]]
    def forward(s, X):
        s.a = [X]
        for k, (W, b) in enumerate(zip(s.W, s.b)):
            z = s.a[-1] @ W + b
            if k < len(s.W) - 1:
                s.a.append(np.maximum(0, z))
            else:
                e = np.exp(z - z.max(1, keepdims=True))
                s.a.append(e / e.sum(1, keepdims=True))
        return s.a[-1]
    def loss(s, X, y):
        P = s.forward(X)
        return -np.log(P[np.arange(len(y)), y]).mean()
    def backward(s, y):
        N = len(y); P = s.a[-1].copy(); P[np.arange(N), y] -= 1; d = P / N
        gW = [None]*len(s.W); gb = [None]*len(s.W)
        for k in reversed(range(len(s.W))):
            gW[k] = s.a[k].T @ d; gb[k] = d.sum(0)
            if k > 0:
                d = (d @ s.W[k].T) * (s.a[k] > 0)
        return gW, gb

In [ ]:
m = MLP([2,16,2])
m.forward(X[:5]); gW, gb = m.backward(y[:5])
eps = 1e-5; W = m.W[0]; i, j = 0, 3
W[i,j] += eps; l1 = m.loss(X[:5], y[:5]); W[i,j] -= 2*eps; l2 = m.loss(X[:5], y[:5]); W[i,j] += eps
print("numeric", (l1-l2)/(2*eps), "analytic", gW[0][i,j])

numeric 0.04671199159700378 analytic 0.046711991595234946


In [ ]:
m = MLP([2,16,2]); lr = 0.5
for ep in range(301):
    m.forward(X); gW, gb = m.backward(y)
    for k in range(len(m.W)):
        m.W[k] -= lr*gW[k]; m.b[k] -= lr*gb[k]
    if ep % 50 == 0:
        P = m.forward(X)
        print(ep, round(m.loss(X, y), 4), "acc", (P.argmax(1) == y).mean())
print("params", sum(w.size for w in m.W) + sum(b.size for b in m.b))

0 0.4415 acc 0.805
50 0.2307 acc 0.895
100 0.1918 acc 0.92
150 0.1486 acc 0.93
200 0.1022 acc 0.97
250 0.0677 acc 0.995
300 0.0471 acc 0.995
params 82


## Chapter recap

## Practice set

**1.** (easy) A single neuron: `z = 2·3 + (−1)·1 + 0.5`. What is `z`, and what is `ReLU(z)`? **Answer:** `z = 6 − 1 + 0.5 = 5.5`; ReLU(5.5) = 5.5.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch02.html